In [ ]:
import pypsa
import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
from _helpers import mock_snakemake
snakemake = mock_snakemake(
            "collect_figures",
            scenario="penalty-oc" # penalty-sa, penalty-ea, penalty-nwa, penalty-oc
    )

In [ ]:
scenario = snakemake.wildcards.scenario

### Paths

In [ ]:
trade_model = f"../../results/cost_year~2030/interone~hbi/intertwo~eaf-grid/final~steel/scenario~{scenario}/network.nc"
# single_region = "../../resources/lcos/cost_year~2030/Europe/network_1.nc" # old
single_region = "../../resources/lco-steel/cost_year~2030/South_South_America/network_1.nc"

### Read trade model


In [ ]:
n = pypsa.Network(trade_model)

### Steel supply curve all regions (from model)

In [ ]:
iron_ore_cost = n.generators[n.generators.carrier == "iron_ore"].marginal_cost.mean()
iron_ore_to_steel = 1/n.links[n.links.carrier == 'steel'].efficiency.mean()
iron_ore_cost_notransport = iron_ore_cost * iron_ore_to_steel
print(f"Iron ore cost (w/o transport): {iron_ore_cost_notransport:.2f} €/t_steel")

In [ ]:
process = "steel"

# Combined steel supply curve (all regions) as stacked area plot by region, sorted by cost
steel_links = n.links[n.links.carrier == 'hbi'].copy()
steel_links['region'] = steel_links['bus1'].str.replace('_hbi$', '', regex=True)
hbi_eaf_cost = n.links[n.links.carrier == 'hbi'].marginal_cost*1.59 + n.links[n.links.carrier == 'steel'].marginal_cost.values
steel_links['total_cost'] = hbi_eaf_cost * iron_ore_to_steel + iron_ore_cost_notransport
steel_links['quantity'] = steel_links['p_nom_max'] / 1.59 / 1e6  # Mt steel

# Sort by cost for supply curve
steel_links_sorted = steel_links.sort_values('total_cost').reset_index(drop=True)

# Assign a color to each region
region_list = steel_links_sorted['region'].unique()
region_colors = snakemake.config["colors"]

cum_quantity_line = steel_links_sorted['quantity'].cumsum()
cum_cost_line = steel_links_sorted['total_cost']

# Prepare for stacked area plot: for each link, plot a bar at its cost, colored by region
cum_quantity = 0
bar_lefts = []
bar_widths = []
bar_costs = []
bar_colors = []
for _, row in steel_links_sorted.iterrows():
    bar_lefts.append(cum_quantity)
    bar_widths.append(row['quantity'])
    bar_costs.append(row['total_cost'])
    bar_colors.append(region_colors[row['region']])
    cum_quantity += row['quantity']

In [ ]:

def plot_supply_curve(bar_lefts, bar_widths, bar_costs, bar_colors, region_colors, region_list):

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(bar_lefts, bar_costs, width=bar_widths, color=bar_colors, align='edge', edgecolor='none', alpha=0.8)
    # Add the classic supply curve line
    
    ax.step(cum_quantity_line, cum_cost_line, where='post', color='black', linewidth=1.2, label='Supply curve')

    # Add vertical line of current demand and REMIND demand
    current_demand = 2000  # Mt steel
    ax.axvline(current_demand, color='red', linestyle='--', label='Current demand (2020)')
    ax.annotate('Current demand (2025)', xy=(current_demand, 50), xytext=(current_demand+50, 150), rotation=90, color='red')
    remind_demand = 725  # Mt steel, REMIND global steel long-term hydrogen: 700-750 Mt
    ax.axvline(remind_demand, color='orange', linestyle='--', label='REMIND 2050 demand')
    ax.annotate('REMIND 2050 demand', xy=(remind_demand, 50), xytext=(remind_demand+50, 150), rotation=90, color='orange')

    ax.set_xlabel('Cumulative quantity (Mt)')
    ax.set_ylabel('Supply cost (€/t_steel)')
    ax.set_title(f'Combined {process} supply curve')
    ax.set_ylim(0, 1000)
    ax.set_xlim(0, sum(bar_widths))
    # ax.set_xlim(0, 2000)
    ax.grid(axis='y', alpha=0.4, zorder=0)
    # Legend for regions
    handles = [plt.Rectangle((0,0),1,1, color=region_colors[reg]) for reg in region_list]
    handles.append(plt.Line2D([0], [0], color='black', linewidth=1.2, label='Supply curve'))
    labels = list(region_list) + ['Supply curve']
    ax.legend(handles, labels, title='Region', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(snakemake.input.global_supply_curve)
    plt.savefig(snakemake.input.global_supply_curve_png, dpi=300)
    plt.show()

    return

In [ ]:
plot_supply_curve(bar_lefts=bar_lefts, bar_widths=bar_widths, bar_costs=bar_costs, bar_colors=bar_colors, region_colors=region_colors, region_list=region_list)

### Steel supply curve (from single curves)

In [ ]:
process = "hbi"
demand = False # Subtract local demand from supply curve
sort_over_all = False # "True" to sort over all regions, "False" to sort within each region
regions = snakemake.config["regions"].keys()

In [ ]:
df_all = pd.DataFrame()

for region in regions:
    
    if demand:
        fn = f"../../resources/supply_curves/cost_year~2030/{region}_{process}.csv"
    else:
        fn = f"../../resources/supply_curves_nodemand/cost_year~2030/{region}_{process}.csv"
    df = pd.read_csv(fn, index_col=0)
    df["region"] = region

    diff = df["demand [t]"].diff()
    diff[0] = diff[1]  # set first value to second to avoid NaN
    df["demand [t]"] = diff


    df_all = pd.concat([df_all, df], ignore_index=True)

if sort_over_all:
    df_all = df_all.sort_values('LCOX [EUR/t]').reset_index(drop=True)
else:
    pass

In [ ]:
cum_quantity_line = df_all['demand [t]'].cumsum()/1e6  # Mt steel
cum_cost_line = df_all['LCOX [EUR/t]']

# Prepare for stacked area plot based on df_all
cum_quantity = 0
bar_lefts = []
bar_widths = []
bar_costs = []
bar_colors = []
for _, row in df_all.iterrows():
    bar_lefts.append(cum_quantity)
    bar_widths.append(row['demand [t]']/1e6)  # Mt steel
    bar_costs.append(row['LCOX [EUR/t]'])
    bar_colors.append(region_colors[row['region']])
    cum_quantity += row['demand [t]']/1e6  # Mt steel

In [ ]:
plot_supply_curve(bar_lefts=bar_lefts, bar_widths=bar_widths, bar_costs=bar_costs, bar_colors=bar_colors, region_colors=region_colors, region_list=region_list)